In [7]:
from ultralytics import YOLO
import cv2
import numpy as np

# 1) Models load karo
det_model = YOLO(r"runs/detect/train/weights/best.pt")          # detection model
cls_model = YOLO(r"runs/classify/train5/weights/best.pt")       # classification model

# 2) Webcam open
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Webcam not accessible")
    exit()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Original frame ka height/width
    h, w = frame.shape[:2]

    # 3) Detection model running
    det_results = det_model(
        frame,
        conf=0.25,
        device=0,
        verbose=False
    )

    det = det_results[0]

    # Annotated frame copy
    out_frame = frame.copy()

    if det.boxes is not None and len(det.boxes) > 0:
        for box in det.boxes:
            # Bounding box coordinates
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

            # ensure bounds safe
            x1 = max(0, min(x1, w - 1))
            x2 = max(0, min(x2, w - 1))
            y1 = max(0, min(y1, h - 1))
            y2 = max(0, min(y2, h - 1))

            # Crop for classification
            crop = frame[y1:y2, x1:x2]
            if crop.size == 0:
                continue

            # 4) Classification model on crop
            cls_results = cls_model(
                crop,
                device=0,
                verbose=False
            )
            r = cls_results[0]
            cls_id = int(r.probs.top1)
            cls_name = cls_model.names[cls_id]
            cls_conf = float(r.probs.data[cls_id])

            # 5) Draw final bbox + label on out_frame
            label = f"{cls_name} {cls_conf:.2f}"
            cv2.rectangle(out_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(
                out_frame,
                label,
                (x1, max(0, y1 - 5)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 0),
                2
            )

    # 6) Show frame
    cv2.imshow("Detection + Classification", out_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


In [5]:
from ultralytics import YOLO
import cv2

# 1) Load classification model
cls_model = YOLO(r"runs/classify/train5/weights/best.pt")

# 2) Webcam open
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Webcam not accessible")
    exit()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # 3) Run classification model on full frame
    cls_results = cls_model(
        frame,
        device=0,
        verbose=False
    )

    # 4) Extract top prediction
    r = cls_results[0]
    cls_id = int(r.probs.top1)
    cls_name = cls_model.names[cls_id]
    cls_conf = float(r.probs.data[cls_id])

    # 5) Annotate frame with classification result
    label = f"{cls_name} {cls_conf:.2f}"
    out_frame = frame.copy()
    cv2.putText(
        out_frame,
        label,
        (10, 30),  # top-left corner
        cv2.FONT_HERSHEY_SIMPLEX,
        1.0,
        (0, 255, 0),
        2
    )

    # 6) Show frame
    cv2.imshow("Real-time Classification", out_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()